# Train XGBoost (Ray Single-Node Multi-GPU)

Single-node multi-GPU XGBoost training using Ray with **local GPU workers**,
Ray Data loading (`read_databricks_tables`), MLflow tracking, and system metrics.

**Derived from:** `train_xgb_ray_gpu.ipynb` (multi-node GPU version)

**Key differences from multi-node GPU notebook:**
1. `ray.init()` instead of `setup_ray_cluster()` — no Spark workers needed
2. Single-node cluster (`num_workers=0`) with multiple GPUs (e.g. NC48ads_A100_v4 has 2× A100)
3. Ray auto-detects all local GPUs — DataParallelTrainer distributes across them
4. `read_databricks_tables` still streams data via SQL Warehouse (no toPandas)
5. No Spark executor deadlock risk — Spark is only used for UC auth, not compute

**Data pipeline:** `ray.data.read_databricks_tables` streams data from SQL Warehouse.
Each Ray worker gets its shard of the data and builds a local DMatrix on GPU.

**Target hardware:**
- NC48ads_A100_v4: 2× A100 80GB, 48 vCPUs, 440GB RAM
- NC96ads_A100_v4: 4× A100 80GB, 96 vCPUs, 880GB RAM

## Setup Widgets

In [ ]:
%pip install -U mlflow pynvml
%restart_python

In [ ]:
# Global error tracking
_notebook_errors = []
def log_error(error_msg, exc=None):
    import traceback
    entry = {"error": str(error_msg)}
    if exc: entry["traceback"] = traceback.format_exc()
    _notebook_errors.append(entry)
    print(f"ERROR LOGGED: {error_msg}")

dbutils.widgets.dropdown("data_size", "tiny", ["tiny", "small", "medium", "medium_large", "large", "xlarge"], "Data Size")
dbutils.widgets.text("node_type", "D8sv5", "Node Type")
dbutils.widgets.dropdown("run_mode", "full", ["full", "smoke"], "Run Mode")
dbutils.widgets.text("num_workers", "0", "Num Workers (0=auto)")
dbutils.widgets.text("cpus_per_worker", "0", "CPUs per Worker (0=auto)")
dbutils.widgets.text("warehouse_id", "148ccb90800933a1", "Databricks SQL Warehouse ID")
dbutils.widgets.text("catalog", "brian_gen_ai", "Catalog")
dbutils.widgets.text("schema", "xgb_scaling", "Schema")
dbutils.widgets.text("table_name", "", "Table Name (override)")
dbutils.widgets.text("max_depth", "8", "Max Tree Depth")
dbutils.widgets.text("num_boost_round", "200", "Boosting Rounds")
dbutils.widgets.text("num_gpus_per_worker", "1", "GPUs per Worker")

In [ ]:
# Get widget values
data_size = dbutils.widgets.get("data_size")
node_type = dbutils.widgets.get("node_type")
run_mode = dbutils.widgets.get("run_mode")
num_workers_input = int(dbutils.widgets.get("num_workers"))
cpus_per_worker_input = int(dbutils.widgets.get("cpus_per_worker"))
warehouse_id = dbutils.widgets.get("warehouse_id").strip()
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
table_name_override = dbutils.widgets.get("table_name").strip()
max_depth_input = int(dbutils.widgets.get("max_depth"))
num_boost_round_input = int(dbutils.widgets.get("num_boost_round"))
num_gpus_per_worker = int(dbutils.widgets.get("num_gpus_per_worker"))

import sys, os
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = "/".join(notebook_path.split("/")[:-2])
sys.path.insert(0, f"/Workspace{repo_root}")
from src.config import PRESETS as CONFIG_PRESETS, get_preset

SIZE_PRESETS = {n: {"suffix": p.table_suffix, "rows": p.rows, "features": p.total_features} for n, p in CONFIG_PRESETS.items()}

if table_name_override:
    input_table = f"{catalog}.{schema}.{table_name_override}"
    data_size_label = table_name_override.replace("imbalanced_", "")
else:
    preset = get_preset(data_size)
    input_table = f"{catalog}.{schema}.imbalanced_{preset.table_suffix}"
    data_size_label = data_size

run_name = f"ray_gpu_smoke_{node_type}" if run_mode == "smoke" else f"ray_gpu_{data_size_label}{'_'+str(num_workers_input)+'w' if num_workers_input > 0 else ''}_{node_type}"
print(f"Config: {data_size} | {node_type} | {run_mode} | table={input_table} | run={run_name}")
print(f"XGB: max_depth={max_depth_input}, num_boost_round={num_boost_round_input}")

# --- Get table schema via SQL Warehouse (avoids spark.table deadlock with Ray) ---
# On GPU ML Runtime, even spark.table() before Ray setup can hang indefinitely.
# Instead, query the warehouse directly via the SQL Statement API.
import json, time, urllib.request

_db_host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().getOrElse(None)
_db_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().getOrElse(None)

def _sql_via_warehouse(sql, wh_id, host=_db_host, token=_db_token, timeout_s=45):
    """Execute SQL via the Databricks SQL Statement API (no Spark needed)."""
    body = json.dumps({"warehouse_id": wh_id, "statement": sql,
                        "wait_timeout": f"{min(timeout_s, 50)}s"}).encode()
    req = urllib.request.Request(
        f"{host}/api/2.0/sql/statements",
        data=body, headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"})
    resp = json.loads(urllib.request.urlopen(req, timeout=timeout_s + 10).read())
    if resp.get("status", {}).get("state") != "SUCCEEDED":
        raise RuntimeError(f"SQL statement failed: {resp}")
    return resp

_desc = _sql_via_warehouse(f"DESCRIBE TABLE {input_table}", warehouse_id)
_string_types = {"STRING", "VARCHAR", "CHAR"}
_numeric_cols = [row[0] for row in _desc["result"]["data_array"]
                 if row[1].upper() not in _string_types and not row[0].startswith("#")]
if "label" not in _numeric_cols:
    raise ValueError(f"Expected 'label' in numeric columns, got: {_numeric_cols}")
print(f"Table schema (via warehouse): {len(_numeric_cols)} numeric columns (skipping string categoricals)")

## Environment Validation Gate

In [ ]:
from src.validate_env import validate_environment
_env_report = validate_environment(
    track="ray-scaling",
    expected_workers=num_workers_input if num_workers_input > 0 else None,
    raise_on_failure=False,
)
if not _env_report.passed:
    print(f"WARNING: {len(_env_report.errors)} validation error(s) — continuing anyway for debugging")


## MLflow Setup

In [ ]:
import os, time

# Enable system metrics logging BEFORE importing mlflow
os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "true"

import mlflow

# Also call the enable function after import
mlflow.enable_system_metrics_logging()

# Use Unity Catalog model registry for model logging/registration
mlflow.set_registry_uri("databricks-uc")
uc_model_name = f"{catalog}.{schema}.xgb_ray_gpu"

# Get current user via dbutils context (no Spark needed)
user_email = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
experiment_path = f"/Users/{user_email}/xgb_scaling_benchmark"

# Set experiment
mlflow.set_experiment(experiment_path)

# --- START MLflow run EARLY for breadcrumb visibility ---
_mlflow_run = mlflow.start_run(run_name=run_name, log_system_metrics=True)
run_id = _mlflow_run.info.run_id
_breadcrumb_t0 = time.time()

# --- Diagnostic log to UC Volume (readable from any session) ---
_diag_log_path = f"/Volumes/{catalog}/{schema}/ray_results/_diag_{run_name}_{run_id[:8]}.log"
def _diag(msg):
    """Write timestamped diagnostic line to UC Volume + MLflow + stdout."""
    elapsed = time.time() - _breadcrumb_t0
    line = f"[{elapsed:7.1f}s] {msg}"
    print(line)
    try:
        with open(_diag_log_path, "a") as f:
            f.write(line + "\n")
    except Exception as e:
        print(f"  (diag write failed: {e})")

def _breadcrumb(label):
    """Log a timestamped breadcrumb param to MLflow for debugging stalls."""
    elapsed = time.time() - _breadcrumb_t0
    mlflow.log_param(f"bc_{label}", f"{elapsed:.1f}s")
    _diag(f"BREADCRUMB: {label}")

_breadcrumb("mlflow_started")
mlflow.log_param("run_name", run_name)
mlflow.log_param("input_table", input_table)
mlflow.log_param("warehouse_id", warehouse_id)
mlflow.log_param("node_type", node_type)
mlflow.log_param("diag_log_path", _diag_log_path)
_diag(f"Diag log: {_diag_log_path}")

print(f"MLflow experiment: {experiment_path}")
print(f"MLflow run (early): {run_id} ({run_name})")
print(f"UC model target: {uc_model_name}")
print(f"System metrics logging enabled: {os.environ.get('MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING')}")

## Initialize Ray (Single-Node)

In [ ]:
import time
import os

_breadcrumb("cell10_ray_setup_start")

# Check Ray version and availability
try:
    import ray
    _diag(f"Ray version: {ray.__version__}")
except ImportError as e:
    raise RuntimeError(f"Ray not available: {e}")

# Setup Databricks environment for Ray workers (required for Ray Data + MLflow)
databricks_host_url = _db_host  # already captured in cell-4
databricks_token = _db_token

# CRITICAL (G9): Strip https:// prefix from HOST.
host_for_ray = databricks_host_url.replace("https://", "").replace("http://", "")
os.environ["DATABRICKS_HOST"] = host_for_ray
os.environ["DATABRICKS_TOKEN"] = databricks_token
_diag(f"Databricks Host (for Ray): {host_for_ray}")

# Get cluster info via Clusters API
import json, urllib.request

_cluster_id = dbutils.notebook.entry_point.getDbutils().notebook().getContext().clusterId().get()
_diag(f"Cluster ID: {_cluster_id}")
_cl_req = urllib.request.Request(
    f"{databricks_host_url}/api/2.0/clusters/get?cluster_id={_cluster_id}",
    headers={"Authorization": f"Bearer {databricks_token}"})
_cl_resp = json.loads(urllib.request.urlopen(_cl_req, timeout=30).read())
num_executors = _cl_resp.get("num_workers", 0)

_diag(f"Cluster workers (Spark): {num_executors} (expected 0 for single-node)")
_breadcrumb("cluster_api_done")
mlflow.log_param("num_executors", num_executors)
mlflow.log_param("cluster_id", _cluster_id)

# Log cluster node type from API response
_actual_node_type = _cl_resp.get("node_type_id", "unknown")
_diag(f"Actual node_type_id from API: {_actual_node_type}")
mlflow.log_param("actual_node_type_id", _actual_node_type)

# Detect GPUs via nvidia-smi BEFORE Ray init
import subprocess
_smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True, timeout=10
)
if _smi.returncode != 0:
    raise RuntimeError("No GPUs detected! Use GPU ML Runtime.")

gpu_lines = _smi.stdout.strip().split('\n')
detected_gpus = len(gpu_lines)
for i, line in enumerate(gpu_lines):
    _diag(f"  GPU {i}: {line.strip()}")
_diag(f"Detected {detected_gpus} GPU(s) on single node")

# Determine per-node vCPU count
import re
node_type_lower = node_type.lower()
node_vcpus_match = re.search(r"[de](\d+)", node_type_lower) or re.search(r"nc(\d+)", node_type_lower)
node_vcpus = int(node_vcpus_match.group(1)) if node_vcpus_match else 48
# Leave some CPUs for Ray head/overhead
allocatable_cpus = max(1, node_vcpus - 2)

# Determine number of Ray training workers (= number of GPUs by default)
if num_workers_input > 0:
    num_workers = min(num_workers_input, detected_gpus)
else:
    num_workers = detected_gpus

# CPUs per worker: distribute allocatable CPUs evenly across GPU workers
if cpus_per_worker_input > 0:
    cpus_per_worker = cpus_per_worker_input
else:
    cpus_per_worker = max(1, allocatable_cpus // num_workers)

_diag(f"Resource sizing: node_vcpus={node_vcpus}, allocatable={allocatable_cpus}")
_diag(f"Ray config: {num_workers}W x {cpus_per_worker}CPU x {num_gpus_per_worker}GPU")
_breadcrumb("cell10_done")

In [ ]:
# Initialize Ray locally (single-node, no setup_ray_cluster)
print("Starting Ray (single-node mode)...")
_breadcrumb("ray_init_start")
ray_start = time.time()

omp_threads_str = str(cpus_per_worker)
os.environ["OMP_NUM_THREADS"] = omp_threads_str

try:
    # ray.init() on a single node auto-detects all CPUs and GPUs.
    # No setup_ray_cluster() needed — that requires Spark workers.
    ray.init(
        runtime_env={
            "env_vars": {
                "OMP_NUM_THREADS": omp_threads_str,
                "DATABRICKS_HOST": host_for_ray,
                "DATABRICKS_TOKEN": databricks_token,
            }
        }
    )
    ray_init_time = time.time() - ray_start
    _breadcrumb("ray_init_done")
    mlflow.log_metric("ray_init_time_sec", ray_init_time)

    # Re-assert driver env vars (G9 note)
    os.environ["DATABRICKS_HOST"] = host_for_ray
    os.environ["DATABRICKS_TOKEN"] = databricks_token

    # --- Dump full Ray cluster state ---
    cluster_resources = ray.cluster_resources()
    available_resources = ray.available_resources()
    _diag(f"Ray cluster_resources: {cluster_resources}")
    _diag(f"Ray available_resources: {available_resources}")
    mlflow.log_param("ray_resources", str(cluster_resources)[:500])

    available_cpus = int(cluster_resources.get("CPU", 0))
    available_gpus = int(cluster_resources.get("GPU", 0))
    mlflow.log_metric("ray_available_cpus", available_cpus)
    mlflow.log_metric("ray_available_gpus", available_gpus)
    _diag(f"Ray CPUs: {available_cpus}, GPUs: {available_gpus}")

    # Dump per-node info (should be just 1 node)
    _diag("--- Ray nodes ---")
    for i, node in enumerate(ray.nodes()):
        alive = node.get("Alive", False)
        node_id = node.get("NodeID", "?")[:12]
        resources = node.get("Resources", {})
        cpu = resources.get("CPU", 0)
        gpu = resources.get("GPU", 0)
        mem_gb = resources.get("memory", 0) / 1e9
        obj_gb = resources.get("object_store_memory", 0) / 1e9
        _diag(f"  node[{i}] {node_id} alive={alive} CPU={cpu} GPU={gpu} mem={mem_gb:.1f}GB obj={obj_gb:.1f}GB")

    # --- GPU-aware resource budget adjustment ---
    max_gpu_workers = available_gpus // max(1, num_gpus_per_worker)
    if num_workers > max_gpu_workers:
        _diag(f"GPU cap: {num_workers}W requested, only {max_gpu_workers} possible with {available_gpus} GPUs. Capping.")
        num_workers = max_gpu_workers

    # Check CPU budget (trainer overhead needs +1 CPU)
    required = num_workers * cpus_per_worker + 1
    _diag(f"Resource budget: {num_workers}W x {cpus_per_worker}CPU = {num_workers*cpus_per_worker} + 1 overhead = {required} needed, {available_cpus} available")
    if required > available_cpus:
        _diag(f"WARNING: Insufficient CPUs! Adjusting...")
        usable = max(1, available_cpus - 1)
        cpus_per_worker = max(1, usable // max(1, num_workers))
        num_workers = min(max(1, usable // max(1, cpus_per_worker)), max_gpu_workers)
        _diag(f"  Adjusted to {num_workers}W x {cpus_per_worker}CPU")

    print(f"Final: {num_workers}W x {cpus_per_worker}CPU x {num_gpus_per_worker}GPU")
    print(f"Ray init time: {ray_init_time:.1f}s")
    _breadcrumb("ray_init_complete")
except Exception as e:
    _breadcrumb("ray_init_FAILED")
    _diag(f"Ray init FAILED: {e}")
    mlflow.log_param("ray_init_error", str(e)[:500])
    import traceback
    _diag(traceback.format_exc())
    traceback.print_exc()
    raise

## Worker-Side System Metrics & OMP Diagnostics

In [ ]:
import ray

@ray.remote(num_cpus=0)
class WorkerMetricsMonitor:
    def __init__(self, run_id, node_id, db_host, db_token, sampling_interval=10.0):
        import os
        os.environ.update({"DATABRICKS_HOST": db_host, "DATABRICKS_TOKEN": db_token, "MLFLOW_TRACKING_URI": "databricks"})
        self._run_id, self._node_id, self._si, self._mon = run_id, node_id, sampling_interval, None
        self._rn = ray.get_runtime_context().get_node_id()[:8]
    def start(self):
        from mlflow.system_metrics.system_metrics_monitor import SystemMetricsMonitor
        self._mon = SystemMetricsMonitor(run_id=self._run_id, node_id=self._node_id, sampling_interval=self._si, samples_before_logging=1)
        self._mon.start(); return f"{self._node_id} on {self._rn}"
    def stop(self):
        if self._mon: self._mon.finish(); self._mon = None; return f"{self._node_id} stopped"
        return f"{self._node_id} n/a"

@ray.remote(num_cpus=0)
class OmpDiagnosticsCollector:
    def __init__(self): self._r = {}
    def report(self, rank, diag): self._r[rank] = diag
    def get_all(self): return dict(self._r)

def start_worker_monitors(run_id, db_host, db_token, si=10.0):
    """Start a single metrics monitor on the head node (single-node mode)."""
    a = WorkerMetricsMonitor.options(name="metrics_head").remote(
        run_id, "head_node", db_host, db_token, si
    )
    result = ray.get(a.start.remote())
    print(f"  {result}")
    return [a]

def stop_worker_monitors(actors):
    if not actors: return
    try:
        for r in ray.get([a.stop.remote() for a in actors], timeout=30): print(f"  {r}")
    except Exception as e: print(f"  WARN: {e}")
    for a in actors:
        try: ray.kill(a)
        except: pass

@ray.remote(num_cpus=0)
class GpuDiagnosticsCollector:
    """Collect GPU info from each worker."""
    def __init__(self): self._r = {}
    def report(self, rank, diag): self._r[rank] = diag
    def get_all(self): return dict(self._r)

print("WorkerMetricsMonitor + OmpDiagnosticsCollector + GpuDiagnosticsCollector defined.")

## Load Data

In [ ]:
import ray.data

_breadcrumb("data_load_start")

if not warehouse_id:
    raise ValueError("warehouse_id is required for distributed Ray Data loading")

# Re-assert driver env vars before read_databricks_tables (G9).
# ray.init(runtime_env=...) only sets env vars for WORKERS, not the driver.
os.environ["DATABRICKS_HOST"] = host_for_ray
os.environ["DATABRICKS_TOKEN"] = databricks_token

print(f"Loading data from: {input_table}")
print(f"Using SQL Warehouse: {warehouse_id}")
load_start = time.time()

# _numeric_cols was computed in the config cell BEFORE Ray claimed Spark executors.
# Using it here avoids a spark.table() call that would deadlock on small-vCPU nodes.
col_list = ", ".join(_numeric_cols)
query = f"SELECT {col_list} FROM {input_table}"
print(f"Query selects {len(_numeric_cols)} numeric columns (skipping StringType categoricals)")

_breadcrumb("read_databricks_tables_call")
# CRITICAL: Pass catalog= and schema= explicitly to avoid internal
# SparkSession.sql("SELECT CURRENT_CATALOG()") call that deadlocks
# after setup_ray_cluster() claims all Spark executors.
full_ray_ds = ray.data.read_databricks_tables(
    warehouse_id=warehouse_id,
    query=query,
    catalog=catalog,
    schema=schema,
)
_breadcrumb("read_databricks_tables_returned")

# Get schema without triggering a full .count() — defer row count to after split
all_columns = list(full_ray_ds.schema().names)
feature_columns = [c for c in all_columns if c != "label"]
# CRITICAL: Pre-sort feature columns for consistent ordering between training and eval.
# sorted() produces lexicographic order (f0, f1, f10, f100...) which differs from
# natural order (f0, f1, f2, f3...). Both training and eval MUST use the same order.
_sorted_feature_cols = sorted(feature_columns)
load_time = time.time() - load_start

_breadcrumb("data_load_done")
mlflow.log_metric("data_load_time_sec", load_time)

print(f"Loaded {len(all_columns)} columns in {load_time:.1f}s")
print(f"Feature count: {len(feature_columns)}")

## Prepare Features and Labels

In [ ]:
# Derive class distribution from preset or known table (avoids expensive full scan).
# Scanning via Ray Data can fail on transient networking issues (DNS, etc.),
# so we avoid it whenever possible.

# First try: direct preset match (when data_size widget selects a preset)
preset_info = get_preset(data_size) if not table_name_override else None

# Second try: match table_name_override against known table names in config
if preset_info is None and table_name_override:
    from src.config import REALISTIC_PRESETS, ALL_PRESETS
    # Check all presets (including realistic_*) for a matching table_name
    for _pname, _preset in {**CONFIG_PRESETS, **REALISTIC_PRESETS}.items():
        if _preset.table_name == table_name_override:
            preset_info = _preset
            print(f"Matched table_name_override '{table_name_override}' to preset '{_pname}'")
            break

if preset_info:
    n_rows = preset_info.rows
    minority_ratio = preset_info.imbalance_ratio
    positive_count = int(n_rows * minority_ratio)
    negative_count = n_rows - positive_count
    print(f"Using preset row count: {n_rows:,} (preset: {preset_info.name})")
else:
    # Custom table — need to compute from data
    n_rows = full_ray_ds.count()
    positive_count = int(full_ray_ds.sum("label"))
    negative_count = n_rows - positive_count
    minority_ratio = positive_count / n_rows if n_rows else 0.0
    print(f"Computed row count: {n_rows:,} (custom table)")

print("Class distribution:")
print(f"  Class 0 (majority): {negative_count:,} ({(negative_count / n_rows) * 100:.2f}%)")
print(f"  Class 1 (minority): {positive_count:,} ({(positive_count / n_rows) * 100:.2f}%)")

# Calculate scale_pos_weight for imbalance
scale_pos_weight = negative_count / max(positive_count, 1)
print(f"\nscale_pos_weight: {scale_pos_weight:.2f}")

In [ ]:
_breadcrumb("split_start")

# Train/test split in Ray Data (keeps ingestion distributed)
split_start = time.time()
train_ray_ds, test_ray_ds = full_ray_ds.train_test_split(test_size=0.2, seed=42)
split_time = time.time() - split_start

_breadcrumb("split_done")
mlflow.log_metric("split_time_sec", split_time)

# Use estimated counts from split ratio to avoid expensive full scans
train_count = int(n_rows * 0.8)
test_count = n_rows - train_count

print(f"Train set: ~{train_count:,} rows (estimated)")
print(f"Test set: ~{test_count:,} rows (estimated)")
print(f"Split time: {split_time:.1f}s")

# Bounded evaluation sample for local sklearn metrics without pandas conversion.
import numpy as np

def _ray_dataset_to_numpy(dataset, feature_cols, label_col="label", batch_size=65536):
    x_batches, y_batches = [], []
    for batch in dataset.iter_batches(batch_format="numpy", batch_size=batch_size):
        x_batch = np.column_stack([batch[c] for c in feature_cols]).astype(np.float32, copy=False)
        y_batch = batch[label_col].astype(np.float32, copy=False)
        x_batches.append(x_batch)
        y_batches.append(y_batch)

    if not x_batches:
        return np.empty((0, len(feature_cols)), dtype=np.float32), np.empty((0,), dtype=np.float32)

    return np.concatenate(x_batches, axis=0), np.concatenate(y_batches, axis=0)

_breadcrumb("eval_sample_start")
eval_sample_rows = min(200_000, test_count)
eval_test_ds = test_ray_ds.limit(eval_sample_rows)
# CRITICAL: Use _sorted_feature_cols (same order as training) to avoid column mismatch.
# sorted() produces lexicographic order (f0, f1, f10, f100...) which differs from
# natural order (f0, f1, f2, f3...). The model trains on sorted order, so eval must match.
X_test_eval, y_test_eval = _ray_dataset_to_numpy(eval_test_ds, _sorted_feature_cols)
_breadcrumb("eval_sample_done")
print(f"Evaluation sample rows: {X_test_eval.shape[0]:,}")

## XGBoost Training (Ray Distributed)

In [ ]:
from ray.train.xgboost import RayTrainReportCallback, XGBoostConfig
from ray.train import ScalingConfig, RunConfig
from ray.train.data_parallel_trainer import DataParallelTrainer
import ray.data, ray.train

xgb_params = {"objective": "binary:logistic", "tree_method": "hist", "device": "cuda",
    "nthread": cpus_per_worker,  # for CPU-side data prep
    "max_depth": max_depth_input, "learning_rate": 0.1, "scale_pos_weight": scale_pos_weight,
    "seed": 42, "verbosity": 1}
num_boost_round = num_boost_round_input
scaling_config = ScalingConfig(num_workers=num_workers, use_gpu=True,
    resources_per_worker={"GPU": num_gpus_per_worker, "CPU": cpus_per_worker})
ray_storage_path = f"/Volumes/{catalog}/{schema}/ray_results/"
os.makedirs(ray_storage_path, exist_ok=True)
run_config = RunConfig(storage_path=ray_storage_path, name="xgb_ray_train")

# Pre-sort feature columns once so workers don't recompute per batch
_sorted_feature_cols = sorted(feature_columns)

def xgb_train_fn(config):
    import os, ctypes
    import numpy as np
    nthread, diag_ref = config.get("nthread", 1), config.get("_omp_diag_ref")
    rank = ray.train.get_context().get_world_rank()
    diag = {"omp_before": os.environ.get("OMP_NUM_THREADS", "NOT_SET")}
    os.environ["OMP_NUM_THREADS"] = str(nthread)
    diag["omp_set_to"] = str(nthread)
    for ln in ["libgomp.so.1", "libomp.so", "libomp.so.5"]:
        try:
            lib = ctypes.CDLL(ln); lib.omp_get_max_threads.restype = ctypes.c_int
            lib.omp_set_num_threads.argtypes = [ctypes.c_int]
            b = lib.omp_get_max_threads(); lib.omp_set_num_threads(nthread)
            diag[f"ctypes_{ln}"] = f"{b}->{lib.omp_get_max_threads()}"
        except OSError: pass
    import xgboost
    for ln in ["libgomp.so.1"]:
        try:
            lib = ctypes.CDLL(ln); lib.omp_get_max_threads.restype = ctypes.c_int
            diag[f"post_{ln}"] = str(lib.omp_get_max_threads())
        except: pass
    if diag_ref:
        try: ray.get(diag_ref.report.remote(rank, diag), timeout=10)
        except: pass

    # GPU diagnostics
    gpu_diag_ref = config.get("_gpu_diag_ref")
    if gpu_diag_ref:
        try:
            import subprocess
            gpu_info = subprocess.run(
                ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,temperature.gpu", "--format=csv,noheader"],
                capture_output=True, text=True, timeout=10
            )
            cuda_visible = os.environ.get("CUDA_VISIBLE_DEVICES", "NOT_SET")
            gpu_diag = {"nvidia_smi": gpu_info.stdout.strip(), "CUDA_VISIBLE_DEVICES": cuda_visible}
            ray.get(gpu_diag_ref.report.remote(rank, gpu_diag), timeout=10)
        except Exception as ge:
            try: ray.get(gpu_diag_ref.report.remote(rank, {"error": str(ge)}), timeout=10)
            except: pass

    label_col, n_rounds = config["label_column"], config["num_boost_round"]
    feat_cols = config.get("_feature_cols")  # Pre-sorted feature column list
    xp = {k: v for k, v in config.items() if k not in ("label_column","num_boost_round","dataset_keys","_omp_diag_ref","_gpu_diag_ref","_feature_cols")}

    def shard_to_dmatrix(shard):
        x_batches, y_batches = [], []
        for batch in shard.iter_batches(batch_format="numpy", batch_size=65536):
            # Use pre-sorted feature columns if provided, else derive from batch
            cols = feat_cols if feat_cols else sorted(c for c in batch.keys() if c != label_col)
            x_batch = np.column_stack([batch[c] for c in cols]).astype(np.float32, copy=False)
            y_batch = batch[label_col].astype(np.float32, copy=False)
            x_batches.append(x_batch)
            y_batches.append(y_batch)
        if not x_batches:
            return xgboost.DMatrix(np.empty((0, 0), dtype=np.float32), label=np.empty((0,), dtype=np.float32))
        X = np.concatenate(x_batches, axis=0)
        y = np.concatenate(y_batches, axis=0)
        return xgboost.DMatrix(X, label=y)

    train_shard = ray.train.get_dataset_shard("train").materialize()
    dtrain = shard_to_dmatrix(train_shard)
    evals = [(dtrain, "train")]
    vds = ray.train.get_dataset_shard("valid")
    if vds:
        evals.append((shard_to_dmatrix(vds.materialize()), "valid"))

    ckpt = ray.train.get_checkpoint()
    sm, iters = None, n_rounds
    if ckpt:
        sm = RayTrainReportCallback.get_model(ckpt)
        iters = n_rounds - sm.num_boosted_rounds()

    xgboost.train(xp, dtrain, evals=evals, num_boost_round=iters, xgb_model=sm, callbacks=[RayTrainReportCallback()])

train_loop_config = {**xgb_params, "label_column": "label", "num_boost_round": num_boost_round,
    "_feature_cols": _sorted_feature_cols}
print(f"XGB: {xgb_params} | rounds={num_boost_round}")
print(f"OMP: spark_conf(L1) + runtime_env(L2) + env+ctypes before import(L3)")
print(f"DataParallelTrainer: {num_workers}W x {cpus_per_worker}CPU x {num_gpus_per_worker}GPU")

In [ ]:
_breadcrumb("training_cell_start")
print(f"Train: ~{train_count:,} | Test: ~{test_count:,}")
_mons, _omp = [], OmpDiagnosticsCollector.remote()
_gpu_diag = GpuDiagnosticsCollector.remote()
train_loop_config["_omp_diag_ref"] = _omp
train_loop_config["_gpu_diag_ref"] = _gpu_diag

# MLflow run was started early (cell-8) for breadcrumb visibility.
print(f"MLflow: {run_id} ({run_name})")
try:
    _mons = start_worker_monitors(run_id, host_for_ray, databricks_token)
    mlflow.log_param("worker_metrics_monitors", len(_mons))
except Exception as e:
    print(f"WARN: monitors failed: {e}"); mlflow.log_param("worker_metrics_monitors", 0)

for k, v in {"training_mode": "ray_multigpu_single_node", "data_size": data_size,
    "run_mode": run_mode, "n_rows": n_rows,
    "n_features": len(feature_columns), "num_workers": num_workers,
    "cpus_per_worker": cpus_per_worker, "num_boost_round": num_boost_round,
    "omp_fix": "runtime_env+env_before_import+ctypes",
    "omp_target": cpus_per_worker,
    "num_gpus_per_worker": num_gpus_per_worker,
    "device": "cuda",
    "single_node": True,
    "detected_gpus": detected_gpus}.items():
    mlflow.log_param(k, v)
for k, v in xgb_params.items(): mlflow.log_param(f"xgb_{k}", v)
mlflow.log_metric("ray_init_time_sec", ray_init_time)

_breadcrumb("trainer_fit_start")
try:
    print(f"\nTraining with DataParallelTrainer (single-node, {num_workers} GPUs)...")
    t0 = time.time()
    trainer = DataParallelTrainer(train_loop_per_worker=xgb_train_fn, train_loop_config=train_loop_config,
        scaling_config=scaling_config, run_config=run_config,
        datasets={"train": train_ray_ds, "valid": test_ray_ds}, backend_config=XGBoostConfig())
    result = trainer.fit()
    train_time = time.time() - t0
    print(f"Done in {train_time:.1f}s"); mlflow.log_metric("train_time_sec", train_time)
    _breadcrumb("trainer_fit_done")
finally:
    stop_worker_monitors(_mons); _mons = []

try:
    od = ray.get(_omp.get_all.remote(), timeout=15)
    print(f"\nOMP diag ({len(od)} workers):")
    for r in sorted(od):
        for k, v in sorted(od[r].items()): print(f"  w{r}/{k}: {v}"); mlflow.log_param(f"omp_w{r}_{k}", str(v)[:500])
except Exception as e: print(f"WARN: OMP diag failed: {e}")

try:
    gd = ray.get(_gpu_diag.get_all.remote(), timeout=15)
    print(f"\nGPU diag ({len(gd)} workers):")
    for r in sorted(gd):
        for k, v in sorted(gd[r].items()): print(f"  w{r}/{k}: {v}"); mlflow.log_param(f"gpu_w{r}_{k}", str(v)[:500])
except Exception as e: print(f"WARN: GPU diag failed: {e}")

import xgboost as xgb
from mlflow.models import infer_signature
t0 = time.time()
booster = RayTrainReportCallback.get_model(result.checkpoint)

# Predict on eval sample for metrics + model signature
yp = booster.predict(xgb.DMatrix(X_test_eval))
y_pred = (yp > 0.5).astype(int)

# Model signature required for Unity Catalog registration (G8)
sig = infer_signature(X_test_eval, yp)
mlflow.xgboost.log_model(
    xgb_model=booster,
    artifact_path="model",
    signature=sig,
    registered_model_name=uc_model_name,
)
mlflow.log_param("registered_model_name", uc_model_name)

pred_time = time.time() - t0; mlflow.log_metric("predict_time_sec", pred_time)

from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
auc_pr, auc_roc = average_precision_score(y_test_eval, yp), roc_auc_score(y_test_eval, yp)
f1 = f1_score(y_test_eval, y_pred)
prec, rec = precision_score(y_test_eval, y_pred, zero_division=0), recall_score(y_test_eval, y_pred, zero_division=0)
for n, v in [("auc_pr",auc_pr),("auc_roc",auc_roc),("f1",f1),("precision",prec),("recall",rec)]:
    mlflow.log_metric(n, v); print(f"  {n}: {v:.4f}")
cm = confusion_matrix(y_test_eval, y_pred)
for n, v in [("true_negatives",cm[0,0]),("false_positives",cm[0,1]),("false_negatives",cm[1,0]),("true_positives",cm[1,1])]:
    mlflow.log_metric(n, v)
print(classification_report(y_test_eval, y_pred, zero_division=0))
total_time = ray_init_time + load_time + split_time + train_time + pred_time
mlflow.log_metric("total_time_sec", total_time)
_breadcrumb("all_done")
print(f"\nDone: {run_name} | {total_time:.1f}s | OMP={cpus_per_worker} | {run_id}")

## Shutdown Ray Cluster

In [ ]:
print("Shutting down Ray...")
ray.shutdown()
print("Ray shutdown complete.")

# End the early-started MLflow run
try:
    mlflow.end_run()
    print("MLflow run ended.")
except:
    pass

## Exit

In [ ]:
import json

# Check if we have all expected variables (indicates successful run)
try:
    result = {
        "status": "ok" if not _notebook_errors else "error",
        "run_name": run_name,
        "run_id": run_id,
        "training_mode": "ray_multigpu_single_node",
        "data_size": data_size,
        "node_type": node_type,
        "warehouse_id": warehouse_id,
        "n_rows": n_rows,
        "n_features": len(feature_columns),
        "num_workers": num_workers,
        "cpus_per_worker": cpus_per_worker,
        "max_depth": max_depth_input,
        "num_boost_round": num_boost_round,
        "num_gpus_per_worker": num_gpus_per_worker,
        "detected_gpus": detected_gpus,
        "device": "cuda",
        "single_node": True,
        "auc_roc": round(auc_roc, 4),
        "auc_pr": round(auc_pr, 4),
        "train_time_sec": round(train_time, 1),
        "total_time_sec": round(total_time, 1),
    }
    if _notebook_errors:
        result["errors"] = _notebook_errors
except NameError as e:
    # Some variables weren't defined - notebook failed early
    result = {
        "status": "error",
        "error": f"Notebook failed before completion: {e}",
        "errors": _notebook_errors if '_notebook_errors' in dir() else [],
    }

result_json = json.dumps(result)
print(f"\nNotebook result: {result_json}")

dbutils.notebook.exit(result_json)